In [ ]:
# In[1]:

import pandas as pd
import pytz
from collections import OrderedDict

tz = pytz.timezone('Asia/Shanghai')

files = ["metric.csv", "trace.csv", "log.csv", "error_logs.csv"]
# mapping file -> key column for "key name"
key_col_map = {
    "metric.csv": "kpi_name",
    "trace.csv": "trace_name",
    "log.csv": "log_name",
    "error_logs.csv": "message"
}

summary_dict = OrderedDict()

for fname in files:
    df = pd.read_csv(fname)
    row_count = int(df.shape[0])
    # distinct cmdb_id count
    if 'cmdb_id' in df.columns:
        distinct_cmdb_id_count = int(df['cmdb_id'].nunique())
        top_cmdb = df['cmdb_id'].value_counts().head(20)
        top_20_cmdb_ids = [{"cmdb_id": idx, "count": int(c)} for idx, c in top_cmdb.items()]
    else:
        distinct_cmdb_id_count = 0
        top_20_cmdb_ids = []
    # key name stats
    key_col = key_col_map[fname]
    if key_col in df.columns:
        distinct_key_name_count = int(df[key_col].nunique())
        top_keys = df[key_col].value_counts().head(20)
        top_20_key_names = [{ "key_name": idx, "count": int(c) } for idx, c in top_keys.items()]
    else:
        distinct_key_name_count = 0
        top_20_key_names = []
    # timestamps
    if 'timestamp' in df.columns and row_count>0:
        ts = pd.to_datetime(df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
        min_timestamp = ts.min().isoformat()
        max_timestamp = ts.max().isoformat()
    else:
        min_timestamp = None
        max_timestamp = None

    summary = {
        "file_name": fname,
        "row_count": row_count,
        "distinct_cmdb_id_count": distinct_cmdb_id_count,
        "top_20_cmdb_ids": top_20_cmdb_ids,
        "distinct_key_name_count": distinct_key_name_count,
        "top_20_key_names": top_20_key_names,
        "min_timestamp": min_timestamp,
        "max_timestamp": max_timestamp
    }
    summary_dict[fname] = summary

summary_dict

```
Out[1]:
```
```python
# Build a plain-English summary from the previously computed summaries (summary_dict)
lines = []
for fname, s in summary_dict.items():
    # top 3 cmdb ids and top 3 key names for compactness
    top_cmdb = s["top_20_cmdb_ids"][:3]
    top_keys = s["top_20_key_names"][:3]
    top_cmdb_str = ", ".join(f'{t["cmdb_id"]} ({t["count"]})' for t in top_cmdb) if top_cmdb else "none"
    top_keys_str = ", ".join(f'{t["key_name"]} ({t["count"]})' for t in top_keys) if top_keys else "none"
    line = (
        f"{fname}: {s['row_count']} rows; {s['distinct_cmdb_id_count']} distinct cmdb_id(s). "
        f"Top CMDBs: {top_cmdb_str}. "
        f"{s['distinct_key_name_count']} distinct key names. Top keys: {top_keys_str}. "
        f"Time range: {s['min_timestamp']} to {s['max_timestamp']}."
    )
    lines.append(line)

summary = "\n".join(lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

OrderedDict([('metric.csv', {'file_name': 'metric.csv', 'row_count': 149652, 'distinct_cmdb_id_count': 59, 'top_20_cmdb_ids': [{'cmdb_id': 'adservice', 'count': 10230}, {'cmdb_id': 'adservice2', 'count': 9990}, {'cmdb_id': 'frontend2-0', 'count': 5280}, {'cmdb_id': 'frontend-0', 'count': 5160}, {'cmdb_id': 'frontend-1', 'count': 5160}, {'cmdb_id': 'frontend-2', 'count': 4980}, {'cmdb_id': 'checkoutservice-0', 'count': 4020}, {'cmdb_id': 'checkoutservice-2', 'count': 4020}, {'cmdb_id': 'checkoutservice-1', 'count': 3780}, {'cmdb_id': 'checkoutservice2-0', 'count': 3540}, {'cmdb_id': 'productcatalogservice-1', 'count': 3480}, {'cmdb_id': 'productcatalogservice-2', 'count': 3480}, {'cmdb_id': 'productcatalogservice-0', 'count': 3480}, {'cmdb_id': 'adservice2-0', 'count': 3060}, {'cmdb_id': 'cartservice-1', 'count': 3000}, {'cmdb_id': 'productcatalogservice2-0', 'count': 2880}, {'cmdb_id': 'cartservice-0', 'count': 2880}, {'cmdb_id': 'cartservice2-0', 'count': 2880}, {'cmdb_id': 'shippingservice-0', 'count': 2640}, {'cmdb_id': 'cartservice-2', 'count': 2640}], 'distinct_key_name_count': 1244, 'top_20_key_names': [{'key_name': 'container.node-5.container_network_transmit_errors.eth0', 'count': 1050}, {'key_name': 'container.node-5.container_network_transmit_packets.eth0', 'count': 1050}, {'key_name': 'container.node-5.container_network_receive_errors.eth0', 'count': 1050}, {'key_name': 'container.node-5.container_network_transmit_packets_dropped.eth0', 'count': 1050}, {'key_name': 'container.node-5.container_network_receive_MB.eth0', 'count': 1050}, {'key_name': 'container.node-5.container_network_receive_packets.eth0', 'count': 1050}, {'key_name': 'container.node-5.container_network_transmit_MB.eth0', 'count': 1050}, {'key_name': 'container.node-5.container_network_receive_packets_dropped.eth0', 'count': 1050}, {'key_name': 'container.node-5.container_tasks_state.iowaiting', 'count': 990}, {'key_name': 'container.node-5.container_tasks_state.stopped', 'count': 990}, {'key_name': 'container.node-5.container_tasks_state.sleeping', 'count': 990}, {'key_name': 'container.node-5.container_tasks_state.running', 'count': 990}, {'key_name': 'container.node-5.container_fs_io_time_seconds./dev/vda1', 'count': 990}, {'key_name': 'container.node-5.container_cpu_cfs_periods', 'count': 990}, {'key_name': 'container.node-5.container_fs_io_time_weighted_seconds./dev/vda1', 'count': 990}, {'key_name': 'container.node-5.container_fs_limit_MB./dev/vda1', 'count': 990}, {'key_name': 'container.node-5.container_fs_read_seconds./dev/vda1', 'count': 990}, {'key_name': 'container.node-5.container_fs_reads./dev/vda1', 'count': 990}, {'key_name': 'container.node-5.container_cpu_cfs_throttled_periods', 'count': 990}, {'key_name': 'container.node-5.container_cpu_cfs_throttled_seconds', 'count': 990}], 'min_timestamp': '2022-03-21T13:00:00+08:00', 'max_timestamp': '2022-03-21T13:29:00+08:00'}), ('trace.csv', {'file_name': 'trace.csv', 'row_count': 33976, 'distinct_cmdb_id_count': 40, 'top_20_cmdb_ids': [{'cmdb_id': 'frontend-1', 'count': 2716}, {'cmdb_id': 'frontend-0', 'count': 2644}, {'cmdb_id': 'frontend-2', 'count': 2512}, {'cmdb_id': 'checkoutservice-1', 'count': 2344}, {'cmdb_id': 'checkoutservice-2', 'count': 2308}, {'cmdb_id': 'checkoutservice-0', 'count': 2128}, {'cmdb_id': 'productcatalogservice-2', 'count': 1164}, {'cmdb_id': 'productcatalogservice-0', 'count': 1164}, {'cmdb_id': 'productcatalogservice-1', 'count': 1156}, {'cmdb_id': 'frontend2-0', 'count': 1080}, {'cmdb_id': 'checkoutservice2-0', 'count': 1080}, {'cmdb_id': 'recommendationservice-0', 'count': 840}, {'cmdb_id': 'recommendationservice-2', 'count': 840}, {'cmdb_id': 'recommendationservice-1', 'count': 840}, {'cmdb_id': 'cartservice-2', 'count': 808}, {'cmdb_id': 'cartservice-1', 'count': 804}, {'cmdb_id': 'cartservice-0', 'count': 796}, {'cmdb_id': 'currencyservice-2', 'count': 700}, {'cmdb_id': 'currencyservice-0', 'count': 700}, {'cmdb_id': 'currencyservice-1', 'count': 696}], 'distinct_key_name_count': 200, 'top_20_key_names': [{'key_name': 'trace.from_frontend-1.row_count', 'count': 619}, {'key_name': 'trace.from_frontend-1.error_rate', 'count': 619}, {'key_name': 'trace.from_frontend-1.duration_p95', 'count': 619}, {'key_name': 'trace.from_frontend-1.duration_mean', 'count': 619}, {'key_name': 'trace.from_frontend-0.row_count', 'count': 601}, {'key_name': 'trace.from_frontend-0.error_rate', 'count': 601}, {'key_name': 'trace.from_frontend-0.duration_p95', 'count': 601}, {'key_name': 'trace.from_frontend-0.duration_mean', 'count': 601}, {'key_name': 'trace.from_frontend-2.duration_mean', 'count': 568}, {'key_name': 'trace.from_frontend-2.duration_p95', 'count': 568}, {'key_name': 'trace.from_frontend-2.error_rate', 'count': 568}, {'key_name': 'trace.from_frontend-2.row_count', 'count': 568}, {'key_name': 'trace.self.duration_mean', 'count': 475}, {'key_name': 'trace.self.duration_p95', 'count': 475}, {'key_name': 'trace.self.error_rate', 'count': 475}, {'key_name': 'trace.self.row_count', 'count': 475}, {'key_name': 'trace.from_checkoutservice-1.duration_mean', 'count': 468}, {'key_name': 'trace.from_checkoutservice-1.duration_p95', 'count': 468}, {'key_name': 'trace.from_checkoutservice-1.error_rate', 'count': 468}, {'key_name': 'trace.from_checkoutservice-1.row_count', 'count': 468}], 'min_timestamp': '2022-03-21T13:00:00+08:00', 'max_timestamp': '2022-03-21T13:29:00+08:00'}), ('log.csv', {'file_name': 'log.csv', 'row_count': 1798, 'distinct_cmdb_id_count': 31, 'top_20_cmdb_ids': [{'cmdb_id': 'adservice-0', 'count': 60}, {'cmdb_id': 'adservice-1', 'count': 60}, {'cmdb_id': 'adservice-2', 'count': 60}, {'cmdb_id': 'cartservice-0', 'count': 60}, {'cmdb_id': 'cartservice-1', 'count': 60}, {'cmdb_id': 'cartservice-2', 'count': 60}, {'cmdb_id': 'checkoutservice-1', 'count': 60}, {'cmdb_id': 'emailservice-2', 'count': 60}, {'cmdb_id': 'currencyservice-0', 'count': 60}, {'cmdb_id': 'currencyservice-1', 'count': 60}, {'cmdb_id': 'currencyservice-2', 'count': 60}, {'cmdb_id': 'emailservice-1', 'count': 60}, {'cmdb_id': 'frontend-2', 'count': 60}, {'cmdb_id': 'frontend-1', 'count': 60}, {'cmdb_id': 'frontend-0', 'count': 60}, {'cmdb_id': 'shippingservice-0', 'count': 60}, {'cmdb_id': 'shippingservice-1', 'count': 60}, {'cmdb_id': 'shippingservice-2', 'count': 60}, {'cmdb_id': 'productcatalogservice-1', 'count': 60}, {'cmdb_id': 'productcatalogservice-0', 'count': 60}], 'distinct_key_name_count': 2, 'top_20_key_names': [{'key_name': 'log.error_count', 'count': 899}, {'key_name': 'log.row_count', 'count': 899}], 'min_timestamp': '2022-03-21T13:00:00+08:00', 'max_timestamp': '2022-03-21T13:29:00+08:00'}), ('error_logs.csv', {'file_name': 'error_logs.csv', 'row_count': 1069, 'distinct_cmdb_id_count': 3, 'top_20_cmdb_ids': [{'cmdb_id': 'frontend-1', 'count': 538}, {'cmdb_id': 'frontend-0', 'count': 378}, {'cmdb_id': 'frontend-2', 'count': 153}], 'distinct_key_name_count': 1, 'top_20_key_names': [{'key_name': 'severity: warning, message: failed to retrieve ads', 'count': 1069}], 'min_timestamp': '2022-03-21T13:00:00+08:00', 'max_timestamp': '2022-03-21T13:29:56+08:00'})])```
```

In [ ]:
# In[2]:

import pandas as pd
import pytz

# reuse variable names if present
tz = pytz.timezone('Asia/Shanghai')

# 1) Load metric.csv
df_metric = pd.read_csv("metric.csv")

# create timezone-aware timestamp column in UTC then convert to Asia/Shanghai (UTC+8)
df_metric['ts'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# 1) Compute global P95 for each (cmdb_id, kpi_name) using entire series (global threshold BEFORE filtering)
df_p95 = (
    df_metric
    .groupby(['cmdb_id', 'kpi_name'], dropna=False)['value']
    .quantile(0.95)
    .reset_index(name='global_P95')
)

# 2) Filter metric.csv to the incident window [2022-03-21T13:00:00+08:00, 2022-03-21T13:30:00+08:00)
start = pd.Timestamp('2022-03-21T13:00:00', tz=tz)
end = pd.Timestamp('2022-03-21T13:30:00', tz=tz)
df_win = df_metric[(df_metric['ts'] >= start) & (df_metric['ts'] < end)].copy()

# merge global_P95 into the windowed data so we can evaluate anomalies
df_win = df_win.merge(df_p95, on=['cmdb_id', 'kpi_name'], how='left')

# 3) Mark anomalies (value > global_P95) and compute required stats per (cmdb_id, kpi_name)
df_win['is_anom'] = df_win['value'] > df_win['global_P95']

# total points and max value in window
agg_base = df_win.groupby(['cmdb_id', 'kpi_name'], dropna=False).agg(
    total_points_in_window = ('value', 'size'),
    max_value_in_window = ('value', 'max')
).reset_index()

# anomaly counts
agg_anom_counts = (
    df_win
    .groupby(['cmdb_id', 'kpi_name'], dropna=False)['is_anom']
    .sum()
    .reset_index(name='anomaly_count_in_window')
)

# earliest anomaly timestamp in window (if any)
df_anom = df_win[df_win['is_anom']]
earliest_anom = (
    df_anom
    .groupby(['cmdb_id', 'kpi_name'], dropna=False)['ts']
    .min()
    .reset_index(name='earliest_anomaly_timestamp_in_window')
)

# combine aggregated results
result = (
    agg_base
    .merge(agg_anom_counts, on=['cmdb_id', 'kpi_name'], how='left')
    .merge(earliest_anom, on=['cmdb_id', 'kpi_name'], how='left')
    .merge(df_p95, on=['cmdb_id', 'kpi_name'], how='left')
)

# fill NaNs for anomaly counts with 0
result['anomaly_count_in_window'] = result['anomaly_count_in_window'].fillna(0).astype(int)

# format earliest_anomaly_timestamp_in_window as ISO strings (or None)
result['earliest_anomaly_timestamp_in_window'] = result['earliest_anomaly_timestamp_in_window'].apply(
    lambda x: x.isoformat() if pd.notnull(x) else None
)

# 4) Filter rows where anomaly_count_in_window > 0, sort by anomaly_count descending, limit to top 20
result_filtered = (
    result[result['anomaly_count_in_window'] > 0]
    .sort_values(by=['anomaly_count_in_window', 'total_points_in_window'], ascending=[False, False])
)

# select and order required columns
result_final = result_filtered[[
    'cmdb_id',
    'kpi_name',
    'global_P95',
    'total_points_in_window',
    'anomaly_count_in_window',
    'earliest_anomaly_timestamp_in_window',
    'max_value_in_window'
]].copy()

# keep the output compact: top 20 rows
result_df = result_final.head(20)

result_df

```
Out[2]:
```
Summary of the metric analysis (compact):

- Returned rows: 20 (top 20 metric series with anomalies).
- Affected component(s): only "adservice" (all returned rows are cmdb_id = adservice).
- Analysis window: 2022-03-21T13:00:00+08:00 through 2022-03-21T13:30:00+08:00.
- Global P95: computed across the entire metric.csv for each (cmdb_id, kpi_name) pair before window filtering.
- Window stats per returned series:
  - total_points_in_window = 30 (one-minute samples over the 30-minute window).
  - anomaly_count_in_window = 2 for every returned series (i.e., 2 points > global_P95).
  - earliest_anomaly_timestamp_in_window ranges across series from 2022-03-21T13:00:00+08:00 up to 2022-03-21T13:28:00+08:00.
  - max_value_in_window (examples) span roughly from 3.37e-03 up to 3.174851e+08 across the listed KPIs.
- Example KPI rows (from the top results):
  - adservice / app.grpc.count — global_P95 ≈ 255.55, anomalies earliest at 2022-03-21T13:09:00+08:00, max ≈ 258.0.
  - adservice / app.grpc.mrt — global_P95 ≈ 0.002463, earliest anomaly 2022-03-21T13:04:00+08:00, max ≈ 0.00337085.
  - adservice / app.grpc.sr — global_P95 ≈ 86.3885, earliest anomaly 2022-03-21T13:00:00+08:00, max ≈ 86.44068.

Interpretation: anomalies during the incident window are concentrated in adservice metrics; they are relatively sparse (only 2 out of 30 points exceed the global P95 for each anomalous KPI). Investigate adservice KPIs listed above (and other adservice JVM/runtime metrics shown) to understand the cause of those brief exceedances.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                           kpi_name    global_P95  total_points_in_window  anomaly_count_in_window earliest_anomaly_timestamp_in_window  max_value_in_window
0    adservice                                     app.grpc.count  2.555500e+02                      30                        2            2022-03-21T13:09:00+08:00         2.580000e+02
1    adservice                                       app.grpc.mrt  2.463254e-03                      30                        2            2022-03-21T13:04:00+08:00         3.370850e-03
3    adservice                                        app.grpc.sr  8.638853e+01                      30                        2            2022-03-21T13:00:00+08:00         8.644068e+01
5    adservice                                       app.http.mrt  9.460312e-03                      30                        2            2022-03-21T13:18:00+08:00         9.525000e-03
13   adservice  runtime.java_lang_Compilation_TotalCompilation...  4.390714e+04                      30                        2            2022-03-21T13:28:00+08:00         4.390800e+04
14   adservice  runtime.java_lang_GarbageCollector_CollectionC...  4.678943e+04                      30                        2            2022-03-21T13:28:00+08:00         4.680208e+04
16   adservice  runtime.java_lang_GarbageCollector_CollectionT...  7.351874e+04                      30                        2            2022-03-21T13:28:00+08:00         7.353842e+04
22   adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...  3.173983e+08                      30                        2            2022-03-21T13:28:00+08:00         3.174851e+08
24   adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...  4.678943e+04                      30                        2            2022-03-21T13:28:00+08:00         4.680208e+04
62   adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...  2.194253e+07                      30                        2            2022-03-21T13:28:00+08:00         2.194317e+07
70   adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...  1.720775e+05                      30                        2            2022-03-21T13:19:00+08:00         1.830313e+05
72   adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...  1.675000e+07                      30                        2            2022-03-21T13:28:00+08:00         1.675014e+07
110  adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...  2.194253e+07                      30                        2            2022-03-21T13:28:00+08:00         2.194317e+07
118  adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...  2.089037e+05                      30                        2            2022-03-21T13:00:00+08:00         2.166347e+05
120  adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...  1.674999e+07                      30                        2            2022-03-21T13:28:00+08:00         1.675014e+07
122  adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...  3.173983e+08                      30                        2            2022-03-21T13:28:00+08:00         3.174851e+08
153  adservice  runtime.java_lang_MemoryPool_CollectionUsage_u...  2.001019e+05                      30                        2            2022-03-21T13:19:00+08:00         2.134980e+05
178  adservice  runtime.java_lang_MemoryPool_PeakUsage_used.Te...  1.675000e+07                      30                        2            2022-03-21T13:28:00+08:00         1.675014e+07
215  adservice  runtime.java_lang_MemoryPool_Usage_used.Code_C...  2.194256e+07                      30                        2            2022-03-21T13:28:00+08:00         2.194317e+07
217  adservice  runtime.java_lang_MemoryPool_Usage_used.Eden_S...  4.847321e+06                      30                        2            2022-03-21T13:01:00+08:00         4.968209e+06```
```

In [ ]:
# In[3]:

import pandas as pd
import pytz

# reuse tz if available, otherwise define
try:
    tz
except NameError:
    tz = pytz.timezone('Asia/Shanghai')

# Load trace.csv
df_trace = pd.read_csv("trace.csv")

# create timezone-aware timestamp column (UTC -> convert to Asia/Shanghai)
df_trace['ts'] = pd.to_datetime(df_trace['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# 1) Compute global P95 for each (cmdb_id, trace_name) using the entire file
df_trace_p95 = (
    df_trace
    .groupby(['cmdb_id', 'trace_name'], dropna=False)['value']
    .quantile(0.95)
    .reset_index(name='global_P95')
)

# 2) Filter to the incident window [2022-03-21T13:00:00+08:00, 2022-03-21T13:30:00+08:00)
start = pd.Timestamp('2022-03-21T13:00:00', tz=tz)
end = pd.Timestamp('2022-03-21T13:30:00', tz=tz)
df_trace_win = df_trace[(df_trace['ts'] >= start) & (df_trace['ts'] < end)].copy()

# 3) Merge global P95 into windowed data and mark anomalies (value > global_P95)
df_trace_win = df_trace_win.merge(df_trace_p95, on=['cmdb_id', 'trace_name'], how='left')
df_trace_win['is_anom'] = df_trace_win['value'] > df_trace_win['global_P95']

# Aggregate required stats per (cmdb_id, trace_name)
agg_base = df_trace_win.groupby(['cmdb_id', 'trace_name'], dropna=False).agg(
    total_points_in_window=('value', 'size'),
    max_value_in_window=('value', 'max')
).reset_index()

agg_anom_counts = df_trace_win.groupby(['cmdb_id', 'trace_name'], dropna=False)['is_anom'] \
    .sum().reset_index(name='anomaly_count_in_window')

df_trace_anom = df_trace_win[df_trace_win['is_anom']]
earliest_anom = df_trace_anom.groupby(['cmdb_id', 'trace_name'], dropna=False)['ts'] \
    .min().reset_index(name='earliest_anomaly_timestamp_in_window')

# Combine results and format
result_trace = (
    agg_base
    .merge(agg_anom_counts, on=['cmdb_id', 'trace_name'], how='left')
    .merge(earliest_anom, on=['cmdb_id', 'trace_name'], how='left')
    .merge(df_trace_p95, on=['cmdb_id', 'trace_name'], how='left')
)

result_trace['anomaly_count_in_window'] = result_trace['anomaly_count_in_window'].fillna(0).astype(int)
result_trace['earliest_anomaly_timestamp_in_window'] = result_trace['earliest_anomaly_timestamp_in_window'] \
    .apply(lambda x: x.isoformat() if pd.notnull(x) else None)

# 4) Filter rows with anomalies, sort and limit to top 20
result_trace_filtered = result_trace[result_trace['anomaly_count_in_window'] > 0] \
    .sort_values(by=['anomaly_count_in_window', 'total_points_in_window'], ascending=[False, False])

result_trace_df = result_trace_filtered[[
    'cmdb_id',
    'trace_name',
    'global_P95',
    'total_points_in_window',
    'anomaly_count_in_window',
    'earliest_anomaly_timestamp_in_window',
    'max_value_in_window'
]].head(20)

# Display compact result
result_trace_df

```
Out[3]:
```
```python
# Plain-English summary string of the trace analysis results
summary = (
    "Trace analysis (top 20 anomalous series, incident window 2022-03-21T13:00:00+08:00 to 2022-03-21T13:30:00+08:00):\n"
    "- Returned rows: 20 (limited to top 20 by anomaly count).\n"
    "- Affected components: primarily adservice (adservice-0, adservice-1, adservice-2, adservice2-0) "
    "and cartservice (cartservice-0, cartservice-1).\n"
    "- Observed trace types: mostly trace.from_frontend-*.{row_count, duration_mean, duration_p95} and trace.self.row_count.\n"
    "- Per-series window stats: total_points_in_window = 30 (30 minutes sampled), anomaly_count_in_window = 2 for each listed series "
    "(i.e., 2 points exceed the global P95). Anomalies are therefore sparse and brief.\n"
    "- Earliest anomaly timestamps (examples) range roughly from 2022-03-21T13:00:00+08:00 up to 2022-03-21T13:23:00+08:00 depending on the series.\n"
    "- Example magnitudes: frontend row_count anomalies show small increases (e.g., counts moving from ~15–22 to observed maxima like 16–23); "
    "duration metrics are very small values (on the order of 1e-5–1e-4 seconds) with brief spikes; cartservice trace.self.row_count peaked up to 71.\n"
    "Interpretation and next steps:\n"
    "1) Anomalies are concentrated in adservice and cartservice traces related to frontend calls — investigate adservice and cartservice pods for the listed timestamps.\n"
    "2) Because each anomalous series has only 2/30 points > global P95, these are short-lived spikes; correlate with logs, error traces, and frontend traffic at those minutes.\n"
    "3) Prioritize adservice (multiple pods) and cartservice traces for root-cause debugging, looking at the specific earliest anomaly minutes noted in the results."
)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                           trace_name  global_P95  total_points_in_window  anomaly_count_in_window earliest_anomaly_timestamp_in_window  max_value_in_window
3     adservice-0      trace.from_frontend-0.row_count   15.550000                      30                        2            2022-03-21T13:23:00+08:00            16.000000
4     adservice-0  trace.from_frontend-1.duration_mean    0.000018                      30                        2            2022-03-21T13:21:00+08:00             0.000032
5     adservice-0   trace.from_frontend-1.duration_p95    0.000030                      30                        2            2022-03-21T13:21:00+08:00             0.000165
7     adservice-0      trace.from_frontend-1.row_count   20.550000                      30                        2            2022-03-21T13:20:00+08:00            21.000000
8     adservice-0  trace.from_frontend-2.duration_mean    0.000019                      30                        2            2022-03-21T13:16:00+08:00             0.000034
9     adservice-0   trace.from_frontend-2.duration_p95    0.000041                      30                        2            2022-03-21T13:16:00+08:00             0.000083
17    adservice-1   trace.from_frontend-1.duration_p95    0.000021                      30                        2            2022-03-21T13:21:00+08:00             0.000023
20    adservice-1  trace.from_frontend-2.duration_mean    0.000017                      30                        2            2022-03-21T13:02:00+08:00             0.000018
21    adservice-1   trace.from_frontend-2.duration_p95    0.000023                      30                        2            2022-03-21T13:02:00+08:00             0.000030
27    adservice-2      trace.from_frontend-0.row_count   15.550000                      30                        2            2022-03-21T13:15:00+08:00            16.000000
33    adservice-2   trace.from_frontend-2.duration_p95    0.000027                      30                        2            2022-03-21T13:04:00+08:00             0.000044
35    adservice-2      trace.from_frontend-2.row_count    6.550000                      30                        2            2022-03-21T13:03:00+08:00             7.000000
37   adservice2-0  trace.from_frontend2-0.duration_p95    0.000028                      30                        2            2022-03-21T13:00:00+08:00             0.000031
52  cartservice-0  trace.from_frontend-0.duration_mean    0.000055                      30                        2            2022-03-21T13:06:00+08:00             0.000056
53  cartservice-0   trace.from_frontend-0.duration_p95    0.000127                      30                        2            2022-03-21T13:06:00+08:00             0.000150
55  cartservice-0      trace.from_frontend-0.row_count   21.550000                      30                        2            2022-03-21T13:11:00+08:00            23.000000
56  cartservice-0  trace.from_frontend-1.duration_mean    0.000041                      30                        2            2022-03-21T13:18:00+08:00             0.000043
63  cartservice-0      trace.from_frontend-2.row_count    8.550000                      30                        2            2022-03-21T13:21:00+08:00            11.000000
67  cartservice-0                 trace.self.row_count   68.550000                      30                        2            2022-03-21T13:15:00+08:00            71.000000
83  cartservice-1      trace.from_frontend-0.row_count   21.550000                      30                        2            2022-03-21T13:11:00+08:00            22.000000```
```

In [ ]:
# In[4]:

import pandas as pd
import pytz

# reuse tz if present
try:
    tz
except NameError:
    tz = pytz.timezone('Asia/Shanghai')

start = pd.Timestamp('2022-03-21T13:00:00', tz=tz)
end = pd.Timestamp('2022-03-21T13:30:00', tz=tz)

# Load error_logs.csv
df_error = pd.read_csv("error_logs.csv")
df_error['ts'] = pd.to_datetime(df_error['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# Filter to incident window
df_error_win = df_error[(df_error['ts'] >= start) & (df_error['ts'] < end)].copy()

# Aggregate per cmdb_id: total count and earliest timestamp
error_agg = (
    df_error_win
    .groupby('cmdb_id', dropna=False)
    .agg(
        total_error_log_count_in_window=('message', 'size'),
        earliest_error_log_timestamp_in_window=('ts', 'min')
    )
    .reset_index()
)

# Format timestamp
error_agg['earliest_error_log_timestamp_in_window'] = error_agg['earliest_error_log_timestamp_in_window'] \
    .apply(lambda x: x.isoformat() if pd.notnull(x) else None)

# Sort by earliest timestamp ascending and limit to top 20
error_summary_df = error_agg.sort_values('earliest_error_log_timestamp_in_window', na_position='last').head(20)

# Load log.csv
df_log = pd.read_csv("log.csv")
df_log['ts'] = pd.to_datetime(df_log['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# Filter to window and log_name == 'log.error_count'
df_log_win = df_log[(df_log['ts'] >= start) & (df_log['ts'] < end) & (df_log['log_name'] == 'log.error_count')].copy()

# Aggregate per cmdb_id
agg_base = (
    df_log_win
    .groupby('cmdb_id', dropna=False)
    .agg(
        total_points_in_window=('value', 'size'),
        max_log_error_count_in_window=('value', 'max')
    )
    .reset_index()
)

# earliest timestamp where log.error_count > 0
df_log_pos = df_log_win[df_log_win['value'] > 0]
earliest_pos = (
    df_log_pos
    .groupby('cmdb_id', dropna=False)['ts']
    .min()
    .reset_index(name='earliest_timestamp_where_log_error_count_gt_0')
)

log_agg = agg_base.merge(earliest_pos, on='cmdb_id', how='left')

# Filter where max_log_error_count_in_window > 0
log_agg = log_agg[log_agg['max_log_error_count_in_window'] > 0].copy()

# Format timestamp
log_agg['earliest_timestamp_where_log_error_count_gt_0'] = log_agg['earliest_timestamp_where_log_error_count_gt_0'] \
    .apply(lambda x: x.isoformat() if pd.notnull(x) else None)

# Sort by earliest_timestamp_where_log_error_count_gt_0 ascending and limit to top 20
log_error_summary_df = log_agg.sort_values('earliest_timestamp_where_log_error_count_gt_0', na_position='last').head(20)

# Prepare JSON-serializable compact outputs (records, limited to top 20 as DataFrames above)
error_logs_summary = error_summary_df.to_dict(orient='records')
log_error_count_summary = log_error_summary_df.to_dict(orient='records')

result = {
    'error_logs_summary': error_logs_summary,
    'log_error_count_summary': log_error_count_summary
}

result

```
Out[4]:
```
```python
summary = (
    "Error-log summary (2022-03-21T13:00:00+08:00 to 2022-03-21T13:30:00+08:00):\n"
    "- frontend-1: 538 error log entries, earliest at 2022-03-21T13:00:00+08:00\n"
    "- frontend-0: 378 error log entries, earliest at 2022-03-21T13:00:01+08:00\n"
    "- frontend-2: 153 error log entries, earliest at 2022-03-21T13:00:05+08:00\n\n"
    "Log.error_count summary (same window):\n"
    "- frontend-0: 30 points, max log.error_count = 16.0, first >0 at 2022-03-21T13:00:00+08:00\n"
    "- frontend-1: 30 points, max log.error_count = 21.0, first >0 at 2022-03-21T13:00:00+08:00\n"
    "- frontend-2: 30 points, max log.error_count = 6.0,  first >0 at 2022-03-21T13:00:00+08:00\n\n"
    "Interpretation: Frontend pods (frontend-0/1/2) show both many error log entries and immediate spikes in log.error_count at the start of the incident window. "
    "Prioritize investigating frontend pods around 13:00+08 to find the root cause (logs, recent deploys, upstream requests)."
)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

{'error_logs_summary': [{'cmdb_id': 'frontend-1', 'total_error_log_count_in_window': 538, 'earliest_error_log_timestamp_in_window': '2022-03-21T13:00:00+08:00'}, {'cmdb_id': 'frontend-0', 'total_error_log_count_in_window': 378, 'earliest_error_log_timestamp_in_window': '2022-03-21T13:00:01+08:00'}, {'cmdb_id': 'frontend-2', 'total_error_log_count_in_window': 153, 'earliest_error_log_timestamp_in_window': '2022-03-21T13:00:05+08:00'}], 'log_error_count_summary': [{'cmdb_id': 'frontend-0', 'total_points_in_window': 30, 'max_log_error_count_in_window': 16.0, 'earliest_timestamp_where_log_error_count_gt_0': '2022-03-21T13:00:00+08:00'}, {'cmdb_id': 'frontend-1', 'total_points_in_window': 30, 'max_log_error_count_in_window': 21.0, 'earliest_timestamp_where_log_error_count_gt_0': '2022-03-21T13:00:00+08:00'}, {'cmdb_id': 'frontend-2', 'total_points_in_window': 30, 'max_log_error_count_in_window': 6.0, 'earliest_timestamp_where_log_error_count_gt_0': '2022-03-21T13:00:00+08:00'}]}```
```

In [ ]:
# In[5]:

import pandas as pd
import pytz

# reuse timezone and dataframes if present, else load
tz = pytz.timezone('Asia/Shanghai')

# time window
start = pd.Timestamp('2022-03-21T13:00:00', tz=tz)
end = pd.Timestamp('2022-03-21T13:30:00', tz=tz)

# --- Metric anomalies for adservice ---
try:
    df_metric
except NameError:
    df_metric = pd.read_csv("metric.csv")

# ensure timestamp column as timezone-aware ts
if 'ts' not in df_metric.columns:
    df_metric['ts'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# 1) global P95 per (cmdb_id, kpi_name) computed on entire series (before filtering)
df_metric_p95 = (
    df_metric
    .groupby(['cmdb_id', 'kpi_name'], dropna=False)['value']
    .quantile(0.95)
    .reset_index(name='global_P95')
)

# 2) filter metric rows to incident window
df_metric_win = df_metric[(df_metric['ts'] >= start) & (df_metric['ts'] < end)].copy()

# restrict to cmdb_id starting with 'adservice'
mask_ad = df_metric_win['cmdb_id'].astype(str).str.startswith('adservice')
df_metric_win_ad = df_metric_win[mask_ad].copy()

# merge global P95
df_metric_win_ad = df_metric_win_ad.merge(df_metric_p95, on=['cmdb_id', 'kpi_name'], how='left')

# find anomalies where value > global_P95
df_metric_anom = df_metric_win_ad[df_metric_win_ad['value'] > df_metric_win_ad['global_P95']].copy()

# total count
metric_anom_count = int(df_metric_anom.shape[0])

# earliest anomaly timestamp and up to 5 rows if tie
if metric_anom_count > 0:
    earliest_ts_metric = df_metric_anom['ts'].min()
    df_metric_earliest = df_metric_anom[df_metric_anom['ts'] == earliest_ts_metric] \
        .sort_values(['cmdb_id','kpi_name']).head(5)
    # format output columns
    df_metric_earliest_out = df_metric_earliest[[
        'cmdb_id','kpi_name','ts','value','global_P95'
    ]].copy()
    df_metric_earliest_out['timestamp'] = df_metric_earliest_out['ts'].apply(lambda x: x.isoformat())
    df_metric_earliest_out = df_metric_earliest_out[[
        'cmdb_id','kpi_name','timestamp','value','global_P95'
    ]]
else:
    df_metric_earliest_out = pd.DataFrame(columns=['cmdb_id','kpi_name','timestamp','value','global_P95'])

# --- Trace anomalies for adservice ---
try:
    df_trace
except NameError:
    df_trace = pd.read_csv("trace.csv")

if 'ts' not in df_trace.columns:
    df_trace['ts'] = pd.to_datetime(df_trace['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# global P95 for traces
df_trace_p95 = (
    df_trace
    .groupby(['cmdb_id', 'trace_name'], dropna=False)['value']
    .quantile(0.95)
    .reset_index(name='global_P95')
)

# window filter and adservice restriction
df_trace_win = df_trace[(df_trace['ts'] >= start) & (df_trace['ts'] < end)].copy()
mask_ad_trace = df_trace_win['cmdb_id'].astype(str).str.startswith('adservice')
df_trace_win_ad = df_trace_win[mask_ad_trace].copy()

# merge p95 and find anomalies
df_trace_win_ad = df_trace_win_ad.merge(df_trace_p95, on=['cmdb_id', 'trace_name'], how='left')
df_trace_anom = df_trace_win_ad[df_trace_win_ad['value'] > df_trace_win_ad['global_P95']].copy()

trace_anom_count = int(df_trace_anom.shape[0])

if trace_anom_count > 0:
    earliest_ts_trace = df_trace_anom['ts'].min()
    df_trace_earliest = df_trace_anom[df_trace_anom['ts'] == earliest_ts_trace] \
        .sort_values(['cmdb_id','trace_name']).head(5)
    df_trace_earliest_out = df_trace_earliest[[
        'cmdb_id','trace_name','ts','value','global_P95'
    ]].copy()
    df_trace_earliest_out['timestamp'] = df_trace_earliest_out['ts'].apply(lambda x: x.isoformat())
    df_trace_earliest_out = df_trace_earliest_out[[
        'cmdb_id','trace_name','timestamp','value','global_P95'
    ]]
else:
    df_trace_earliest_out = pd.DataFrame(columns=['cmdb_id','trace_name','timestamp','value','global_P95'])

# --- Earliest frontend error log within window ---
try:
    df_error
except NameError:
    df_error = pd.read_csv("error_logs.csv")

if 'ts' not in df_error.columns:
    df_error['ts'] = pd.to_datetime(df_error['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

df_error_win = df_error[(df_error['ts'] >= start) & (df_error['ts'] < end)].copy()
mask_front = df_error_win['cmdb_id'].astype(str).str.startswith('frontend')
df_error_front = df_error_win[mask_front].copy()

if not df_error_front.empty:
    earliest_front_ts = df_error_front['ts'].min()
    df_front_earliest = df_error_front[df_error_front['ts'] == earliest_front_ts].sort_values('cmdb_id').head(1)
    df_front_earliest_out = df_front_earliest[['cmdb_id','ts','message']].copy()
    df_front_earliest_out['timestamp'] = df_front_earliest_out['ts'].apply(lambda x: x.isoformat())
    df_front_earliest_out = df_front_earliest_out[['cmdb_id','timestamp','message']]
else:
    df_front_earliest_out = pd.DataFrame(columns=['cmdb_id','timestamp','message'])

# --- Prepare compact JSON-like result (limit total rows) ---
# Convert DataFrames to list-of-dicts (compact), but keep small sizes as requested
adservice_metric_earliest = df_metric_earliest_out.to_dict(orient='records')
adservice_trace_earliest = df_trace_earliest_out.to_dict(orient='records')
frontend_earliest_error_log = df_front_earliest_out.to_dict(orient='records')

counts = {
    'adservice_metric_anomaly_count_in_window': metric_anom_count,
    'adservice_trace_anomaly_count_in_window': trace_anom_count
}

result = {
    'adservice_metric_earliest_anomalies': adservice_metric_earliest,   # up to 5 rows
    'adservice_trace_earliest_anomalies': adservice_trace_earliest,     # up to 5 rows
    'frontend_earliest_error_log': frontend_earliest_error_log,         # single row (if any)
    'counts': counts
}

result

```
Out[5]:
```
Summary (chronology and counts):

- Earliest adservice metric anomalies (occurring at 2022-03-21T13:00:00+08:00):
  - adservice / app.grpc.sr (value 86.4407 > P95 86.3885)
  - adservice / runtime.java_lang_GarbageCollector_LastGcInfo_memoryUsageBeforeGc_used.Survivor_Space.Copy (value ~210,287 > P95 ~208,903)
  - adservice / runtime.java_lang_OperatingSystem_SystemCpuLoad (value 0.01793527 > P95 0.01793515)
  - adservice / runtime.jvm_gc_collection_seconds.Copy (value ~0.01400000000001 > P95 ~0.01400000000000)
  - adservice-2 / mesh.destination.frontend.adservice.istio_request_duration_milliseconds.grpc.200.0.0 (value ~0.10147 > P95 ~0.09974)

- Earliest adservice trace anomalies (also at 2022-03-21T13:00:00+08:00):
  - adservice2-0 / trace.from_frontend2-0.duration_mean (2.0e-05 > P95 1.9e-05)
  - adservice2-0 / trace.from_frontend2-0.duration_p95 (2.8e-05 > P95 2.755e-05)

- Earliest frontend error log in the window:
  - frontend-1 at 2022-03-21T13:00:00+08:00 — message: "severity: warning, message: failed to retrieve ads"

- Summary counts (incident window 13:00–13:30 +08):
  - Total adservice metric anomaly points: 434
  - Total adservice trace anomaly points: 38

Interpretation / next steps:
- Metric and trace anomalies for adservice (including adservice2-0 and adservice-2) and the frontend error log all appear at the very start of the incident window (13:00+08), suggesting a correlated event.
- Key areas to investigate first: adservice JVM/GC activity, system CPU load, Istio/request latency for frontend→adservice calls, and frontend failures to retrieve ads at ~13:00.
- Correlate these timestamps with deployment/activity logs and frontend request patterns around 13:00 to find the root cause.

The original code execution output of IPython Kernel is also provided below for reference:

{'adservice_metric_earliest_anomalies': [{'cmdb_id': 'adservice', 'kpi_name': 'app.grpc.sr', 'timestamp': '2022-03-21T13:00:00+08:00', 'value': 86.4406779661017, 'global_P95': 86.38852672750978}, {'cmdb_id': 'adservice', 'kpi_name': 'runtime.java_lang_GarbageCollector_LastGcInfo_memoryUsageBeforeGc_used.Survivor_Space.Copy', 'timestamp': '2022-03-21T13:00:00+08:00', 'value': 210287.33333333337, 'global_P95': 208903.73333333334}, {'cmdb_id': 'adservice', 'kpi_name': 'runtime.java_lang_OperatingSystem_SystemCpuLoad', 'timestamp': '2022-03-21T13:00:00+08:00', 'value': 0.0179352682925363, 'global_P95': 0.017935148205380943}, {'cmdb_id': 'adservice', 'kpi_name': 'runtime.jvm_gc_collection_seconds.Copy', 'timestamp': '2022-03-21T13:00:00+08:00', 'value': 0.01400000000001, 'global_P95': 0.014000000000003565}, {'cmdb_id': 'adservice-2', 'kpi_name': 'mesh.destination.frontend.adservice.istio_request_duration_milliseconds.grpc.200.0.0', 'timestamp': '2022-03-21T13:00:00+08:00', 'value': 0.1014749999999767, 'global_P95': 0.09974249999993474}], 'adservice_trace_earliest_anomalies': [{'cmdb_id': 'adservice2-0', 'trace_name': 'trace.from_frontend2-0.duration_mean', 'timestamp': '2022-03-21T13:00:00+08:00', 'value': 2e-05, 'global_P95': 1.9e-05}, {'cmdb_id': 'adservice2-0', 'trace_name': 'trace.from_frontend2-0.duration_p95', 'timestamp': '2022-03-21T13:00:00+08:00', 'value': 2.8e-05, 'global_P95': 2.7549999999999995e-05}], 'frontend_earliest_error_log': [{'cmdb_id': 'frontend-1', 'timestamp': '2022-03-21T13:00:00+08:00', 'message': 'severity: warning, message: failed to retrieve ads'}], 'counts': {'adservice_metric_anomaly_count_in_window': 434, 'adservice_trace_anomaly_count_in_window': 38}}```
```